In [3]:
# setup + imports
import sys
sys.path.append('..')

import pandas as pd
import re
from pathlib import Path
from config import engine

def extract_uid(filename):
    """Extract uid from filenames like calendar_u01.csv -> u01"""
    match = re.search(r'u\d+', filename)
    return match.group() if match else None

print("✓ Setup complete")


# define path to calendar folder
project_root = Path.cwd().parent
folder = project_root / "dataset" / "archive (1)" / "dataset" / "calendar"

print("Project root:", project_root)
print("Dataset folder:", folder)
print("Folder exists:", folder.exists())


# read all csv files and combine into one dataframe
dfs = []

for file in folder.glob("*.csv"):
    uid = extract_uid(file.name)

    temp_df = pd.read_csv(file)
    temp_df["uid"] = uid

    dfs.append(temp_df)

df = pd.concat(dfs, ignore_index=True)

print("✓ Raw dataframe created")
print("Shape:", df.shape)
print("DataFrame columns:", df.columns.tolist())
display(df.head())


# clean column names
df.columns = df.columns.str.strip()

# read SQL table columns
table_cols_df = pd.read_sql("""
SELECT column_name
FROM information_schema.columns
WHERE table_schema = 'public'
  AND table_name = 'calendar'
ORDER BY ordinal_position;
""", engine)

table_cols = table_cols_df["column_name"].tolist()

print("SQL table columns:", table_cols)


# do not manually insert auto-generated id columns
auto_id_cols = {"calendar_id", "id"}

insert_cols = [col for col in table_cols if col in df.columns and col not in auto_id_cols]

df_to_insert = df[insert_cols].copy()

print("Columns to insert:", df_to_insert.columns.tolist())
print("Shape to insert:", df_to_insert.shape)
display(df_to_insert.head())


# insert into SQL table
df_to_insert.to_sql(
    "calendar",
    engine,
    schema="public",
    if_exists="append",
    index=False
)

print("✓ Data inserted into table 'calendar'")


# verification
check_df = pd.read_sql("SELECT * FROM public.calendar LIMIT 10;", engine)
check_df

✓ Setup complete
Project root: d:\uni\databases\project
Dataset folder: d:\uni\databases\project\dataset\archive (1)\dataset\calendar
Folder exists: True
✓ Raw dataframe created
Shape: (1687, 7)
DataFrame columns: ['id', 'device', 'timestamp', 'ACCOUNT_LABEL', 'DATE', 'TIME', 'uid']


,id,device,timestamp,ACCOUNT_LABEL,DATE,TIME,uid
0,6a9d4195-9193-47f0-a674-59e19ee5b29a-29,1977b545-a88f-4903-a7ae-2c434de4be49,1364182283,1,3/24/2013,20:00,u00
1,3ade29d5-98f5-4a6c-bcbe-25e278a0bc73-55,1977b545-a88f-4903-a7ae-2c434de4be49,1364268683,1,3/25/2013,21:00,u00
2,3ade29d5-98f5-4a6c-bcbe-25e278a0bc73-56,1977b545-a88f-4903-a7ae-2c434de4be49,1364268683,2,3/25/2013,22:00,u00
3,3ade29d5-98f5-4a6c-bcbe-25e278a0bc73-57,1977b545-a88f-4903-a7ae-2c434de4be49,1364268683,3,3/25/2013,15:30,u00
4,bea2978d-ca53-4e32-89ba-19b45e64a897-59,1977b545-a88f-4903-a7ae-2c434de4be49,1364355083,1,3/26/2013,15:30,u00


SQL table columns: ['event_id', 'uid', 'record_id', 'device', 'timestamp', 'account_label', 'date', 'time']
Columns to insert: ['uid', 'device', 'timestamp']
Shape to insert: (1687, 3)


,uid,device,timestamp
0,u00,1977b545-a88f-4903-a7ae-2c434de4be49,1364182283
1,u00,1977b545-a88f-4903-a7ae-2c434de4be49,1364268683
2,u00,1977b545-a88f-4903-a7ae-2c434de4be49,1364268683
3,u00,1977b545-a88f-4903-a7ae-2c434de4be49,1364268683
4,u00,1977b545-a88f-4903-a7ae-2c434de4be49,1364355083


✓ Data inserted into table 'calendar'


,event_id,uid,record_id,device,timestamp,account_label,date,time
0,1,u00,None,1977b545-a88f-4903-a7ae-2c434de4be49,1364182283,None,None,None
1,2,u00,None,1977b545-a88f-4903-a7ae-2c434de4be49,1364268683,None,None,None
2,3,u00,None,1977b545-a88f-4903-a7ae-2c434de4be49,1364268683,None,None,None
3,4,u00,None,1977b545-a88f-4903-a7ae-2c434de4be49,1364268683,None,None,None
4,5,u00,None,1977b545-a88f-4903-a7ae-2c434de4be49,1364355083,None,None,None
5,6,u00,None,1977b545-a88f-4903-a7ae-2c434de4be49,1364355083,None,None,None
6,7,u00,None,1977b545-a88f-4903-a7ae-2c434de4be49,1364355083,None,None,None
7,8,u00,None,1977b545-a88f-4903-a7ae-2c434de4be49,1364355083,None,None,None
8,9,u00,None,1977b545-a88f-4903-a7ae-2c434de4be49,1364355083,None,None,None
9,10,u00,None,1977b545-a88f-4903-a7ae-2c434de4be49,1364355083,None,None,None


In [ ]:
#change